# Flow Matching Tutorial with Gaussian and Independent Coupling Models

Below is a self-contained Jupyter notebook that walks you through implementing and validating two continuous flow matching (CFM) models on a 1D toy dataset and on MNIST. It includes:

- Data setup for 1D and MNIST  
- Definition and training of  
  - Gaussian CFM  
  - Independent Coupling CFM  
- Euler solver for generation  
- Validation  
  - Histogram comparison for 1D  
  - Sample visualization for MNIST  

## 1- Introduction

Flow matching frames generative modeling as matching a time-indexed vector field to a known velocity field that pushes samples from a simple prior toward the data distribution.  

We’ll implement two variants:  
- Gaussian CFM, where the conditional is Gaussian with mean $t·x_1$ and evolving variance  
- Independent Coupling CFM, which couples a data sample and a prior via a simple vector field  

Each model is trained by minimizing the mean-squared error between a learned vector field $v_\theta(t, x)$ and a known target velocity $u_t$.  

Dependencies

In [ ]:
import torch
from torch import nn, Tensor
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
import torch.nn.functional as F
from torch.autograd import grad
from torchdiffeq import odeint
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
import math

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 2- One-Dimensional Mixture of Gaussians dataset

### We define a Mixture of Gaussians as our 1D dataset

In [ ]:
def Mixture_Gaussians(n_samples, n_mg_components, mg_means, mg_var, mg_weights):
   # Step 1: Choose components based on weights
    component_choices = np.random.choice(n_mg_components, size=n_samples, p=mg_weights)
    # Step 2: Sample from selected Gaussians
    x = np.array([
        np.random.normal(mg_means[i], np.sqrt(mg_var[i])) for i in component_choices
    ])

    return x

Let's plot the histogram to get a better feel for the data

In [ ]:
n_mg_components = 2
mg_means = np.array([-6, 6])  # Means of Gaussians
mg_var = np.array([0.4, 0.4])     # Variances
mg_weights = np.array([0.5, 0.5])  # Mixture weights
n_samples = 5000 # Number of samples
x_1D = Mixture_Gaussians(n_samples, n_mg_components, mg_means, mg_var, mg_weights)

plt.hist(x_1D, bins=50)
plt.title("Histogram of Mixture of Gaussians distribution");



Next, we need to define a neural network for the vector field. We use a MLP for the 1D dataset.

In [ ]:
class MLPVectorField(nn.Module):
    def __init__(self, dim: int = 1, h: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, h), nn.ELU(),
            nn.Linear(h, h), nn.ELU(),
            nn.Linear(h, dim))
    
    def forward(self, t: Tensor, x_t: Tensor) -> Tensor:
        if t.dim() == 0:
            t = t.expand_as(x_t)

        return self.net(torch.cat((t, x_t), -1))
    
    def generation(self, x, n_euler_steps, t_start=0.0, t_end=1.0):
        time_steps = torch.linspace(t_start, t_end, n_euler_steps + 1).to(x.device)

        for i in range(n_euler_steps):
            x = x + (time_steps[i+1]- time_steps[i]) * self(t=time_steps[i], x_t=x)    

        return x    

### Independent Coupling Flow Matching (ICFM)

$p_t(x|z) = \mathcal{N}(x \mid (1-t)x_0 + t x_1, \sigma^2 I)$

$u_t(x|z) = x_1 - x_0$

Now, we implement the training function for our first flow matching model

In [ ]:
def ICFM_training(VF, n_epochs, sigma):
    optimizer = torch.optim.Adam(VF.parameters(), 1e-2)
    loss_fn = nn.MSELoss()
    loss_hist = []
    for epoch in range(n_epochs):
        x_1 = Tensor(x_1D).unsqueeze(-1)
        x_0 = torch.randn_like(x_1)
        t = torch.rand(len(x_1), 1)
        x_t = (1 - t) * x_0 + t * x_1 + torch.randn_like(x_0) * sigma
        u = x_1 - x_0
        
        optimizer.zero_grad()
        loss = loss_fn(VF(t=t, x_t=x_t), u)
        loss.backward()
        optimizer.step()
        loss_hist.append(loss.item())
        if epoch % 100 == 0:
            print('epoch: ', epoch, ', loss: ', loss.item())

    return VF, loss_hist    

Let's train our first flow matching model

In [ ]:
VF = MLPVectorField(dim=1)
VF_ICFM, loss_hist_ICFM = ICFM_training(VF, n_epochs=200, sigma=0.1)
plt.plot(loss_hist_ICFM)
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Training loss');

We randomly generate 5000 samples from the trained model and plot it's histogram to evaluate the model

In [ ]:
x_prior = torch.randn(5000, 1)
n_steps = 8
time_steps = torch.linspace(0, 1.0, n_steps + 1)
x_gen_ICFM = VF_ICFM.generation(x=x_prior, n_euler_steps=n_steps)

plt.hist(x_gen_ICFM.detach().numpy(), bins=50)
plt.title('Histogram of Independent Coupling Flow Matching learned distribution');

### Gaussian Flow Matching (GFM)

$p_t(x|x_1) = \mathcal{N}(x \mid t x_1, (t\sigma - t + 1)^2 I)$

$u_t(x|x_1) = \frac{x_1 - (1-\sigma)x}{1 - (1-\sigma)t}$

Here, we implement the training function of our second flow matching model

In [ ]:
def GFM_training(VF, n_epochs, sigma):
    optimizer = torch.optim.Adam(VF.parameters(), 1e-2)
    loss_fn = nn.MSELoss()
    loss_hist = []
    for epoch in range(n_epochs):
        x_1 = Tensor(x_1D).unsqueeze(-1)
        t = torch.rand(len(x_1), 1)
        x_t = t * x_1 +  torch.randn_like(t * x_1) * (t * sigma - t + 1).abs() 
        u = (x_1 - (1-sigma)*x_t) / (1-(1-sigma)*t)
        
        optimizer.zero_grad()
        loss = loss_fn(VF(t=t, x_t=x_t), u)
        loss.backward()
        optimizer.step()
        loss_hist.append(loss.item())
        if epoch % 100 == 0:
            print('epoch: ', epoch, ', loss: ', loss.item())

    return VF, loss_hist    

training the Gaussian Flow Matching model

In [ ]:
VF = MLPVectorField(dim=1)
VF_GFM, loss_hist_GFM = GFM_training(VF, n_epochs=200, sigma=0.1)
plt.plot(loss_hist_GFM)
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Training loss');

Evaluating the trained flow matching model

In [ ]:
x_prior = torch.randn(5000, 1)
n_steps = 8
time_steps = torch.linspace(0, 1.0, n_steps + 1)
x_gen_GFM = VF_GFM.generation(x=x_prior, n_euler_steps=n_steps)

plt.hist(x_gen_GFM.detach().numpy(), bins=50)
plt.title('Histogram of Gaussian Flow Matching learned distribution');
 

## 3- MNIST dataset

### As the second dataset, we now use the MNIST handwritten image dataset.

We download the MNIST data and show a few of its images

In [ ]:
# Define transform: ToTensor + Flatten
transform = transforms.Compose([
    transforms.ToTensor(),
])

mnist_train = datasets.MNIST(root="data", train=True, download=True, transform=transform)
mnist_train_loader = DataLoader(mnist_train, batch_size=128, shuffle=True)



fig, axes = plt.subplots(4, 4, figsize=(4, 4))
for i in range(16):
    img, label = mnist_train[i]  
    axes[i // 4, i % 4].imshow(img.squeeze(), cmap="gray")
    axes[i // 4, i % 4].axis("off")

plt.tight_layout()
plt.show()


Here, we define a Convolutional Neural Network for the vector field

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1)
        self.norm1 = nn.GroupNorm(8 if out_ch>=8 else 1, out_ch)
        self.norm2 = nn.GroupNorm(8 if out_ch>=8 else 1, out_ch)
        if in_ch != out_ch:
            self.nin = nn.Conv2d(in_ch, out_ch, kernel_size=1)
        else:
            self.nin = nn.Identity()

    def forward(self, x):
        h = self.conv1(x)
        h = self.norm1(h)
        h = F.relu(h)
        h = self.conv2(h)
        h = self.norm2(h)
        out = F.relu(h + self.nin(x))
        return out

# -------------------------
# UNet-like architecture (with t as extra channel)
# -------------------------
class UNet(nn.Module):
    def __init__(self, in_ch=1, base_ch=64):
        super().__init__()
        # encoder
        self.enc1 = ResBlock(in_ch+1, base_ch)    # +1 channel for t
        self.enc2 = ResBlock(base_ch, base_ch*2)
        self.enc3 = ResBlock(base_ch*2, base_ch*4)
        # decoder
        self.dec3 = ResBlock(base_ch*4 + base_ch*2, base_ch*2)
        self.dec2 = ResBlock(base_ch*2 + base_ch, base_ch)
        self.out_conv = nn.Sequential(
            nn.Conv2d(base_ch, base_ch, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(base_ch, in_ch, kernel_size=1)
        )
        self.pool = nn.AvgPool2d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='nearest')

    def forward(self, t, x):
        # x: (B, C, H, W), t: (B, 1, 1, 1) in [0,1]
        B, _, H, W = x.shape
        # expand t to (B,1,H,W)
        t_map = t.expand(B,1,H,W)
        xt = torch.cat([x, t_map], dim=1)   # concat along channel
        # encoder
        e1 = self.enc1(xt)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        # decoder
        d3 = self.upsample(e3)
        d3 = torch.cat([d3, e2], dim=1)
        d3 = self.dec3(d3)
        d2 = self.upsample(d3)
        d2 = torch.cat([d2, e1], dim=1)
        d2 = self.dec2(d2)
        out = self.out_conv(d2)
        return out
    

    def generation(self, x, n_euler_steps, t_start=0.0, t_end=1.0):
        time_steps = torch.linspace(t_start, t_end, n_euler_steps + 1).view(-1,1,1,1).to(x.device)

        for i in range(n_euler_steps):
            x = x + (time_steps[i+1]- time_steps[i]) * self(t=time_steps[i], x=x)

        return x    


### Independent Coupling Flow Matching (ICFM)

$p_t(x|z) = \mathcal{N}(x \mid (1-t)x_0 + t x_1, \sigma^2 I)$

$u_t(x|z) = x_1 - x_0$

In [ ]:
def ICFM_MNIST_training(VF, n_epochs, sigma):
    optimizer = torch.optim.Adam(VF.parameters(), 1e-2)
    loss_fn = nn.MSELoss()
    loss_hist = []
    for epoch in range(n_epochs):
        for (x, _) in mnist_train_loader:
            x_1 = x.to(device)   # (B, 1, 28, 28)
            x_0 = torch.randn_like(x_1).to(device)
            t = torch.rand(len(x_1), 1).to(device)   # (B,)
            t = t.view(-1,1,1,1)    # (B, 1, 1, 1)
            x_t = (1 - t) * x_0 + t * x_1 + torch.randn_like(x_0) * sigma
            u = x_1 - x_0     # (B, 1, 28, 28)
            
            optimizer.zero_grad()
            loss = loss_fn(VF(t=t, x=x_t), u)
            loss.backward()
            optimizer.step()
        if epoch % 5 == 0:
            print('epoch: ', epoch, ', loss: ', loss.item())
        loss_hist.append(loss.item())    

    return VF, loss_hist    

In [ ]:
VF = UNet().to(device)
VF_ICFM, loss_hist_ICFM = ICFM_MNIST_training(VF, n_epochs=1, sigma=0.1)
plt.plot(loss_hist_ICFM)
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Training loss');

Finally, we generate random images from the trained Independent Coupling Flow Matching Model

In [ ]:
x_prior = torch.randn(16, 1, 28, 28).to(device)
n_steps = 5
x_gen_ICFM = VF_ICFM.generation(x=x_prior, n_euler_steps=n_steps) 

x_gen_ICFM = x_gen_ICFM.cpu().detach().numpy()
fig, axes = plt.subplots(4, 4, figsize=(4, 4))
for i in range(16):
    img = x_gen_ICFM[i]  
    axes[i // 4, i % 4].imshow(img.squeeze(), cmap="gray")
    axes[i // 4, i % 4].axis("off")

plt.tight_layout()
plt.show()

### Gaussian Flow Matching (GFM)

$p_t(x|x_1) = \mathcal{N}(x \mid t x_1, (t\sigma - t + 1)^2 I)$

$u_t(x|x_1) = \frac{x_1 - (1-\sigma)x}{1 - (1-\sigma)t}$


In this section, you are asked to write the training function of Gaussian Flow Matching for MNIST dataset. In your code, you should train the VF and update the loss history list and return them as outputs (similar to previous training functions). 

In [ ]:
def GFM_MNIST_training(VF, n_epochs, sigma):
    optimizer = torch.optim.Adam(VF.parameters(), 1e-2)
    loss_fn = nn.MSELoss()
    loss_hist = []
    for epoch in range(n_epochs):
        for (x, _) in mnist_train_loader:
            '''

            
            write your code here



            '''



    return VF, loss_hist    

In [ ]:
VF = UNet().to(device)
VF_GFM, loss_hist_GFM = GFM_MNIST_training(VF, n_epochs=10, sigma=0.1)
plt.plot(loss_hist_GFM)
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Training loss');

In [ ]:
x_prior = torch.randn(16, 1, 28, 28).to(device)
n_steps = 5
x_gen_GFM = VF_GFM.generation(x=x_prior, n_euler_steps=n_steps) 

x_gen_GFM = x_gen_GFM.cpu().detach().numpy()
fig, axes = plt.subplots(4, 4, figsize=(4, 4))
for i in range(16):
    img = x_gen_GFM[i]  
    axes[i // 4, i % 4].imshow(img.squeeze(), cmap="gray")
    axes[i // 4, i % 4].axis("off")

plt.tight_layout()
plt.show()

I hope you enjoyed working through this notebook and that it gave you a good start for exploring flow matching on your own.